# Traffic Demand Prediction 

This notebook validates `sample_submission.csv` and defines a reusable submission checker.

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'dataset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_PATH = DATA_DIR / 'sample_submission.csv'

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/underxcore/Desktop/Flipkart_GridLock


In [2]:
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
display(sample)
print('sample shape:', sample.shape)
print('test shape:', test.shape)

,Index,demand
0,0,0.0908
1,1,0.0899
2,2,0.0070
3,3,0.0791
4,4,0.0546


sample shape: (5, 2)
test shape: (41778, 10)


## Expected Submission Contract

In [3]:
EXPECTED_COLUMNS = ['Index', 'demand']
EXPECTED_ROWS = len(test)
print('Expected columns:', EXPECTED_COLUMNS)
print('Expected row count:', EXPECTED_ROWS)

Expected columns: ['Index', 'demand']
Expected row count: 41778


In [4]:
def validate_submission(submission: pd.DataFrame, test_df: pd.DataFrame) -> None:
    assert list(submission.columns) == ['Index', 'demand'], 'Submission columns must be exactly Index,demand'
    assert len(submission) == len(test_df), 'Submission row count must match test.csv'
    assert submission['Index'].equals(test_df['Index']), 'Index values/order must match test.csv'
    assert submission['demand'].notna().all(), 'Demand predictions cannot contain missing values'
    assert np.isfinite(submission['demand']).all(), 'Demand predictions must be finite'
    assert submission['demand'].between(0, 1).all(), 'Demand predictions should be clipped to [0, 1]'
    print('Submission validation passed.')

# Demonstration with a constant baseline matching the required shape.
baseline = pd.DataFrame({'Index': test['Index'], 'demand': 0.1})
validate_submission(baseline, test)
baseline.head()

Submission validation passed.


,Index,demand
0,0,0.1000
1,1,0.1000
2,2,0.1000
3,3,0.1000
4,4,0.1000


In [5]:
baseline_path = OUTPUT_DIR / 'baseline_submission.csv'
baseline.to_csv(baseline_path, index=False)
print(f'Wrote {baseline_path} with shape {baseline.shape}')

Wrote /Users/underxcore/Desktop/Flipkart_GridLock/outputs/baseline_submission.csv with shape (41778, 2)


## Submission Notes

- The platform expects exactly 41,778 rows and two columns: `Index` and `demand`.
- The generated modeling submission should preserve the test file index order exactly.